# Challenge 2 — N₂ ground state with VQE, optimized

**Goal.** Maximize

$$\text{Score} = \text{round}\Big(\text{Base}(N)\times A^2\times \frac{1}{1+0.25\,N_{\text{swap}}}\times \frac{2N}{\max(D,2N)}\Big)$$

with $N \ge 6$, where $\text{Base}(N)=3000+50(N^2-36)$ and
$A=\max(0,1-|E_{\text{vqe}}-E_{\text{exact}}|/0.5)$.

**What the starter notebook does.** `real_amplitudes(reps=1, entanglement='linear')`
on 6 qubits with COBYLA(200): **2,641 points**. Changing only `num_qubits = 8`
gives **2,796** — worse than preparing the Hartree–Fock state alone (**4,001**).

**What the ground state actually is.** The 8-qubit exact state is

$$|\psi_0\rangle \approx 0.9914\,|11110000\rangle + 0.1280\,|11101000\rangle + \dots$$

i.e. **98.29% Hartree–Fock** plus **1.64% of a single q3→q4 excitation**.
The entanglement entropy across the middle cut is only **0.125 bits**. This is a
weakly correlated state, not a strongly correlated one.

**Consequences.**

1. A deep hardware-efficient ansatz is unnecessary — and actively harmful on hardware.
2. A 6-layer nearest-neighbour brickwall reaches **4,392 / 4,400** noiselessly.
3. A **2-CX Givens rotation** reaches **4,386** noiselessly and **4,145** on
   `ibm_marrakesh` — the best noisy score by a wide margin.

| Circuit | N | 2q gates | Noiseless error | Noiseless score | Hardware score |
|---|---|---|---|---|---|
| HF only | 8 | 0 | 2.32e-2 | 4,001 | — |
| 6-layer brickwall | 8 | 21 CX | 4.41e-4 | **4,392** | 251 |
| 2-layer brickwall | 8 | 7 CX | 7.75e-4 | 4,386 | 2,233 |
| 2-CX Givens | 8 | 2 CX | 7.75e-4 | 4,386 | **4,145** |

## How the score works

- **Base** grows quadratically with $N$: 6 → 3,000; 8 → 4,400; 16 → 14,000.
- **Accuracy** is squared. At the challenge's reference error of 0.5 Ha,
  chemical accuracy (1.6 mHa) gives $A^2=0.9936$; 0.44 mHa gives $A^2=0.9982$.
- **SWAP** costs 25% per gate. Nearest-neighbour CXs on a line → zero SWAPs.
- **Depth** is free up to $D=2N$. At $N=8$ the budget is 16; the brickwall uses 13.

The starter leaves most of the depth budget unused *and* the ansatz cannot
express the tiny correlation, so COBYLA just drifts off HF and returns.

In [ ]:
# Run once if needed:
# %pip install -q qiskit qiskit-aer matplotlib requests scipy numpy

import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

print("imports OK")

## 1. Setup — the algorithm module

The algorithm module is embedded below so this notebook is self-contained.

- `brickwall_ansatz(n, layers)` — RY layer + alternating **even/odd** CX layers.
  Unlike `real_amplitudes(entanglement='linear')`, the CXs are parallel, so the
  depth is $1+2\,\text{layers}$ instead of a sequential ladder of length $N-1$.
- `givens_ansatz(n)` — HF preparation + `CX(a,b) · RY(θ,a) · CX(a,b)`, a
  particle-number-conserving rotation between the highest occupied and lowest
  unoccupied orbital.
- `solve_brickwall`, `solve_givens`, `score_parts`, `preview_score`,
  `transpile_linear`, `to_qasm` / `from_qasm`.

In [ ]:
"""N2 VQE — Challenge 2 (intermediate), optimized.

What this module is for
-----------------------
The starter notebook uses `real_amplitudes(reps=1, entanglement='linear')`
on 6 qubits and COBYLA(200).  That scores 2,641.  Simply changing N to 8
scores 2,796 -- worse than preparing the Hartree-Fock state alone (4,001).

The exact 8-qubit ground state is not strongly correlated:

    P(|11110000>) = 0.9829          (Hartree-Fock determinant)
    P(|11101000>) = 0.0164          (single excitation q3 -> q4)
    entanglement entropy ~ 0.125 bits

So the right circuits are:

* `brickwall_ansatz(8, 6)` -- nearest-neighbour even/odd CX layers.
  Depth 13 <= 2N, zero SWAPs, error 4.41e-4 Ha -> 4,392 / 4,400.
* `givens_ansatz(8)` -- a single particle-hole rotation between q3 and q4.
  Depth 4, 2 CX, error 7.75e-4 Ha -> 4,386 noiseless, 4,145 on
  ibm_marrakesh (the 6-layer circuit scores only 251 on hardware).

The scoring formula implemented here is the one from the challenge:

    Base(N) = 3000 + 50 * (N^2 - 36)
    A       = max(0, 1 - |E_vqe - E_exact| / 0.5)
    S_swap  = 1 / (1 + 0.25 * N_swap)
    D_depth = 2N / max(depth, 2N)
    Score   = round(Base * A^2 * S_swap * D_depth)
"""

import os
from pathlib import Path

import numpy as np
from scipy.optimize import minimize
from qiskit import QuantumCircuit, qasm2, transpile
from qiskit.circuit import Parameter, ParameterVector
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.transpiler import CouplingMap

BASELINE_ERROR = 0.5  # Hartree, the challenge's reference error
KEY_NAMES = ("IBM_Key", "IBM_Crn", "Team_Key", "Submission_URL")


# ---------------------------------------------------------------------------
# Hamiltonians
# ---------------------------------------------------------------------------

def get_n2_hamiltonian_6q() -> SparsePauliOp:
    """N2 6-qubit valence active-space Hamiltonian at equilibrium."""
    return SparsePauliOp.from_list([
        ('IIIIII', -107.5000),
        ('ZIIIII', -0.8500), ('IZIIII', -0.8500), ('IIZIII', -0.6200),
        ('IIIZII',  0.4500), ('IIIIZI',  0.4500), ('IIIIIZ',  0.7800),
        ('ZZIIII',  0.1800), ('IZZIII',  0.1800), ('IIZZII',  0.1800),
        ('IIIZZI',  0.1800), ('IIIIZZ',  0.1800),
        ('XXIIII',  0.0900), ('IXXIII',  0.0900), ('IIXXII',  0.0900),
        ('IIIXXI',  0.0900), ('IIIIXX',  0.0900),
        ('YYIIII',  0.0900), ('IYYIII',  0.0900), ('IIYYII',  0.0900),
        ('IIIYYI',  0.0900), ('IIIIYY',  0.0900),
        ('ZIIIIZ',  0.1200),
    ])


def get_n2_hamiltonian_8q() -> SparsePauliOp:
    """N2 8-qubit full p-orbital active-space Hamiltonian."""
    return SparsePauliOp.from_list([
        ('IIIIIIII', -107.5000),
        ('ZIIIIIII', -0.8500), ('IZIIIIII', -0.8500), ('IIZIIIII', -0.6200), ('IIIZIIII', -0.6200),
        ('IIIIZIII',  0.4500), ('IIIIIZII',  0.4500), ('IIIIIIZI',  0.7800), ('IIIIIIIZ',  0.7800),
        ('ZZIIIIII',  0.1800), ('IZZIIIII',  0.1800), ('IIZZIIII',  0.1800),
        ('IIIZZIII',  0.1800), ('IIIIZZII',  0.1800), ('IIIIIZZI',  0.1800), ('IIIIIIZZ',  0.1800),
        ('XXIIIIII',  0.0900), ('IXXIIIII',  0.0900), ('IIXXIIII',  0.0900),
        ('IIIXXIII',  0.0900), ('IIIIXXII',  0.0900), ('IIIIIXXI',  0.0900), ('IIIIIIXX',  0.0900),
        ('YYIIIIII',  0.0900), ('IYYIIIII',  0.0900), ('IIYYIIII',  0.0900),
        ('IIIYYIII',  0.0900), ('IIIIYYII',  0.0900), ('IIIIIYYI',  0.0900), ('IIIIIIYY',  0.0900),
        ('ZIIIIIIZ',  0.1200),
    ])


# ---------------------------------------------------------------------------
# Energies and reference state
# ---------------------------------------------------------------------------

def exact_energy(hamiltonian: SparsePauliOp) -> float:
    """Lowest eigenvalue of the Hamiltonian (full diagonalisation)."""
    return float(np.linalg.eigh(hamiltonian.to_matrix())[0][0])


def hf_energy(hamiltonian: SparsePauliOp) -> float:
    """Energy of the Hartree-Fock determinant: first N/2 qubits occupied."""
    n = hamiltonian.num_qubits
    index = sum(1 << q for q in range(n // 2))
    state = np.zeros(2 ** n)
    state[index] = 1.0
    return float(np.real(state.conj() @ hamiltonian.to_matrix() @ state))


def hf_initial_params(nparams: int, n: int, seed: int = 42, noise: float = 0.05) -> np.ndarray:
    """First N/2 RY angles at pi (occupied), the rest near zero, plus jitter."""
    params = np.zeros(nparams)
    params[: n // 2] = np.pi
    rng = np.random.default_rng(seed)
    return params + rng.uniform(-noise, noise, nparams)


def statevector_energy(circuit: QuantumCircuit, hamiltonian: SparsePauliOp,
                       params=None) -> float:
    """Exact <psi|H|psi> for a (possibly parameterised) circuit."""
    bound = circuit.assign_parameters(params) if params is not None else circuit
    state = Statevector.from_instruction(bound)
    return float(np.real(state.expectation_value(hamiltonian)))


# ---------------------------------------------------------------------------
# Ansätze
# ---------------------------------------------------------------------------

def brickwall_ansatz(n: int, layers: int):
    """RY layer + alternating even/odd nearest-neighbour CX layers.

    This is the shallow linear topology the starter notebook *wanted*:
    every CX is between adjacent qubits, so it transpiles to a line with
    zero SWAPs.  Unlike `real_amplitudes(entanglement='linear')`, the CX
    layers are parallel (depth 2 per layer instead of a sequential ladder).
    """
    qc = QuantumCircuit(n)
    pv = ParameterVector("t", n * (layers + 1))
    k = 0
    for q in range(n):
        qc.ry(pv[k], q)
        k += 1
    for layer in range(layers):
        for q in range(layer % 2, n - 1, 2):
            qc.cx(q, q + 1)
        for q in range(n):
            qc.ry(pv[k], q)
            k += 1
    return qc, pv


def givens_ansatz(n: int, pair=None, occupied=None):
    """Hartree-Fock preparation + one particle-hole (Givens) rotation.

    The rotation is  CX(a,b) . RY(theta,a) . CX(a,b)  which mixes
    |1_a 0_b> and |0_a 1_b> without changing the particle number.
    For N2 the dominant correction to HF is exactly q3 -> q4.
    """
    if pair is None:
        pair = (n // 2 - 1, n // 2)
    if occupied is None:
        occupied = range(n // 2)
    a, b = pair
    qc = QuantumCircuit(n)
    qc.x(list(occupied))
    qc.cx(a, b)
    theta = Parameter("theta")
    qc.ry(theta, a)
    qc.cx(a, b)
    return qc, theta


# ---------------------------------------------------------------------------
# Optimisation
# ---------------------------------------------------------------------------

def optimize(circuit: QuantumCircuit, hamiltonian: SparsePauliOp,
             initial_params, maxiter: int = 2500, rhobeg: float = 0.1):
    """COBYLA on the exact statevector energy."""
    result = minimize(
        lambda p: statevector_energy(circuit, hamiltonian, p),
        initial_params,
        method="COBYLA",
        options={"maxiter": maxiter, "rhobeg": rhobeg},
    )
    return result


def _solve(circuit: QuantumCircuit, hamiltonian: SparsePauliOp, params) -> dict:
    exact = exact_energy(hamiltonian)
    energy = statevector_energy(circuit, hamiltonian, params)
    bound = circuit.assign_parameters(params)
    metrics = circuit_metrics(bound)
    parts = score_parts(metrics["num_qubits"], abs(energy - exact),
                        metrics["depth"], metrics["swaps"])
    return {
        "circuit": bound,
        "params": np.asarray(params, dtype=float),
        "energy": energy,
        "exact": exact,
        "error": abs(energy - exact),
        "metrics": metrics,
        "parts": parts,
        "score": parts["score"],
    }


def solve_brickwall(hamiltonian: SparsePauliOp, layers: int = 6,
                    maxiter: int = 2500, seed: int = 42) -> dict:
    """Optimise the even/odd brickwall.  6 layers -> 4,392 at N=8."""
    n = hamiltonian.num_qubits
    circuit, pv = brickwall_ansatz(n, layers)
    result = optimize(circuit, hamiltonian,
                      hf_initial_params(len(pv), n, seed=seed),
                      maxiter=maxiter)
    return _solve(circuit, hamiltonian, result.x)


def solve_givens(hamiltonian: SparsePauliOp, pair=None, seed: int = 42) -> dict:
    """Optimise the single Givens angle (1 parameter).  4,386 at N=8."""
    n = hamiltonian.num_qubits
    circuit, theta = givens_ansatz(n, pair=pair)
    grid = np.linspace(0.0, 2.0 * np.pi, 2001)
    energies = [statevector_energy(circuit, hamiltonian, [t]) for t in grid]
    start = float(grid[int(np.argmin(energies))])
    result = minimize(
        lambda p: statevector_energy(circuit, hamiltonian, p),
        [start],
        method="Nelder-Mead",
    )
    return _solve(circuit, hamiltonian, result.x)


# ---------------------------------------------------------------------------
# Scoring and circuit utilities
# ---------------------------------------------------------------------------

def score_parts(n: int, error: float, depth: int, swaps: int = 0,
                baseline_error: float = BASELINE_ERROR) -> dict:
    """Breakdown of the challenge score."""
    accuracy = max(0.0, 1.0 - error / baseline_error)
    base = 3000.0 + 50.0 * (n ** 2 - 36)
    swap_factor = 1.0 / (1.0 + 0.25 * swaps)
    depth_factor = (2 * n) / max(depth, 2 * n)
    score = round(base * accuracy ** 2 * swap_factor * depth_factor)
    return {
        "base": base,
        "accuracy": accuracy,
        "accuracy_factor": accuracy ** 2,
        "swap_factor": swap_factor,
        "depth_factor": depth_factor,
        "score": score,
    }


def linear_coupling_map(n: int) -> CouplingMap:
    """Bidirectional line 0 - 1 - ... - (n-1)."""
    edges = []
    for i in range(n - 1):
        edges.extend([(i, i + 1), (i + 1, i)])
    return CouplingMap(edges)


def transpile_linear(circuit: QuantumCircuit, n: int | None = None,
                     optimization_level: int = 3, seed: int = 42) -> QuantumCircuit:
    """Transpile onto a linear coupling map (the hardware the score assumes)."""
    n = n or circuit.num_qubits
    return transpile(circuit, coupling_map=linear_coupling_map(n),
                     optimization_level=optimization_level,
                     seed_transpiler=seed)


def circuit_metrics(circuit: QuantumCircuit) -> dict:
    ops = dict(circuit.count_ops())
    return {
        "num_qubits": circuit.num_qubits,
        "depth": circuit.depth(),
        "swaps": ops.get("swap", 0),
        "ops": ops,
    }


def to_qasm(circuit: QuantumCircuit) -> str:
    return qasm2.dumps(circuit)


def from_qasm(text: str) -> QuantumCircuit:
    return qasm2.loads(text)


def preview_score(circuit: QuantumCircuit, vqe_energy: float, exact: float,
                  baseline_error: float = BASELINE_ERROR) -> int:
    """Print the same breakdown as the challenge's local previewer."""
    metrics = circuit_metrics(circuit)
    parts = score_parts(metrics["num_qubits"], abs(vqe_energy - exact),
                        metrics["depth"], metrics["swaps"], baseline_error)
    print("=" * 56)
    print(f"SCORE BREAKDOWN — N2 ({metrics['num_qubits']} qubits)")
    print("=" * 56)
    print(f"  Base potential:        {parts['base']:,.0f} pts")
    print(f"  Absolute energy error: {abs(vqe_energy - exact):.6e} Ha")
    print(f"  Accuracy A:            {parts['accuracy']:.6f} "
          f"(A^2 = {parts['accuracy_factor']:.6f})")
    print(f"  SWAP gates:            {metrics['swaps']} "
          f"(factor {parts['swap_factor']:.3f})")
    print(f"  Depth:                 {metrics['depth']} / budget {2 * metrics['num_qubits']} "
          f"(factor {parts['depth_factor']:.3f})")
    print("-" * 56)
    print(f"  PROJECTED SCORE:       {parts['score']:,} POINTS")
    print("=" * 56)
    return parts["score"]


def load_keys(path: str | Path | None = None) -> dict:
    """Load ``KEY=VALUE`` secrets from a gitignored file.

    With no argument, search order is:

    1. ``$KEYS_FILE``
    2. ``<repo>/.env``            (the canonical, gitignored file)
    3. ``<repo>/keys.txt``
    4. ``<repo>/Begginer/keys.txt``
    5. ``<repo>/Intermidiate/keys.txt``
    6. ``./.env`` and ``./keys.txt``

    Real environment variables override values read from the file, so CI can
    inject secrets without creating a file at all.
    """
    if path is not None:
        candidates = [Path(path)]
    else:
        candidates = []
        env_path = os.environ.get("KEYS_FILE")
        if env_path:
            candidates.append(Path(env_path))
        try:
            here = Path(__file__).resolve().parent
        except NameError:  # embedded in a notebook: no __file__
            here = Path.cwd().resolve()
        repo = here.parent if here.name in ("Begginer", "Intermidiate") else here
        candidates += [
            repo / ".env",
            repo / "keys.txt",
            repo / "Begginer" / "keys.txt",
            repo / "Intermidiate" / "keys.txt",
            Path.cwd() / ".env",
            Path.cwd() / "keys.txt",
        ]

    keys = {}
    for candidate in candidates:
        if not candidate.is_file():
            continue
        for line in candidate.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            key = key.strip()
            value = value.strip().strip('"').strip("'")
            if key:
                keys[key] = value
        break

    for key in set(keys) | set(KEY_NAMES):
        if os.environ.get(key):
            keys[key] = os.environ[key]
    return keys


# ---------------------------------------------------------------------------
# Reference numbers (used by tests and the notebook)
# ---------------------------------------------------------------------------

REFERENCE = {
    "exact_6q": -111.1031118833,
    "exact_8q": -112.1432365180,
    "hf_8q": -112.1200000000,
    "brickwall_6l_energy": -112.1427952383,
    "brickwall_6l_score": 4392,
    "brickwall_2l_energy": -112.1424620442,
    "brickwall_2l_score": 4386,
    "givens_energy": -112.1424616031,
    "givens_score": 4386,
}

## 2. The two N₂ Hamiltonians and their exact energies

The 6-qubit and 8-qubit Hamiltonians are the ones shipped with the challenge.
`exact_energy` diagonalises the full $2^N \times 2^N$ matrix — fine for $N \le 12$.

In [ ]:
H6 = get_n2_hamiltonian_6q()
H8 = get_n2_hamiltonian_8q()

for name, H in (("6q", H6), ("8q", H8)):
    exact = exact_energy(H)
    ehf = hf_energy(H)
    print(f"{name}: exact = {exact:.10f} Ha   HF = {ehf:.10f} Ha   "
          f"HF error = {ehf - exact:.3e} Ha")

exact8 = exact_energy(H8)
print(f"\nN=8 Hartree-Fock alone scores "
      f"{score_parts(8, hf_energy(H8) - exact8, 1)['score']:,} points")

## 3. The submitted circuit — 6-layer brickwall, 4,392 points

The QASM embedded in the next cell is exactly what was sent to
`challenge_intermediate`. It has 8 qubits, 56 RY + 21 CX, depth 13, zero SWAPs.

In [ ]:
BEST_QASM = """OPENQASM 2.0;
include "qelib1.inc";
qreg q[8];
ry(3.1414241998357997) q[0];
ry(3.4935801312812367) q[1];
ry(3.1414933754366983) q[2];
ry(2.7444615276436335) q[3];
ry(0.021195923257369135) q[4];
ry(0.004390232926535319) q[5];
ry(0.15043368980514482) q[6];
ry(1.0418274964438317) q[7];
cx q[0],q[1];
cx q[2],q[3];
cx q[4],q[5];
cx q[6],q[7];
ry(0.0821136701461508) q[0];
ry(-0.2894983056681932) q[1];
ry(0.04786094853669441) q[2];
ry(-0.3985606862270474) q[3];
ry(-0.28695803050569485) q[4];
ry(-0.1861962432446644) q[5];
ry(1.2698916627464312) q[6];
ry(0.6765801855482059) q[7];
cx q[1],q[2];
cx q[3],q[4];
cx q[5],q[6];
ry(-0.08206954981768455) q[0];
ry(-0.14083079259765452) q[1];
ry(0.02345591478730797) q[2];
ry(-1.5755223418374562) q[3];
ry(0.28386046432651213) q[4];
ry(-0.12023719381791606) q[5];
ry(0.6533040275313632) q[6];
ry(-0.12138917641566938) q[7];
cx q[0],q[1];
cx q[2],q[3];
cx q[4],q[5];
cx q[6],q[7];
ry(0.14882547202858412) q[0];
ry(-0.2661182654357046) q[1];
ry(-0.16146042641381733) q[2];
ry(-0.8977718235391462) q[3];
ry(-0.04302521572698126) q[4];
ry(-0.651347284334642) q[5];
ry(-0.5118497734766828) q[6];
ry(0.050896222357379305) q[7];
cx q[1],q[2];
cx q[3],q[4];
cx q[5],q[6];
ry(-0.14853956458092077) q[0];
ry(-0.4400381827725722) q[1];
ry(-0.18586085645910294) q[2];
ry(0.5932887616743082) q[3];
ry(-0.021276424681811294) q[4];
ry(-0.5681796800369873) q[5];
ry(-0.5180100704887335) q[6];
ry(0.025605737469040254) q[7];
cx q[0],q[1];
cx q[2],q[3];
cx q[4],q[5];
cx q[6],q[7];
ry(0.32792634406411597) q[0];
ry(1.100092893740867) q[1];
ry(1.570873921223596) q[2];
ry(-0.40558127978252967) q[3];
ry(-0.12932971911774158) q[4];
ry(0.699582957747114) q[5];
ry(0.56267035247554) q[6];
ry(-0.525346652866437) q[7];
cx q[1],q[2];
cx q[3],q[4];
cx q[5],q[6];
ry(-0.3286417585851727) q[0];
ry(0.9456905916453812) q[1];
ry(1.5566093123669915) q[2];
ry(0.6541175200678488) q[3];
ry(-0.11011782683360206) q[4];
ry(0.9104223919112131) q[5];
ry(-1.575131557974826) q[6];
ry(-1.0356670521725184) q[7];"""
submitted = from_qasm(BEST_QASM)
submitted_energy = statevector_energy(submitted, H8)
submitted_error = abs(submitted_energy - exact8)
submitted_metrics = circuit_metrics(submitted)
submitted_parts = score_parts(8, submitted_error, submitted_metrics["depth"],
                              submitted_metrics["swaps"])

print(f"energy = {submitted_energy:.10f} Ha")
print(f"exact  = {exact8:.10f} Ha")
print(f"error  = {submitted_error:.6e} Ha")
print(f"depth  = {submitted_metrics['depth']} (budget 16), "
      f"swaps = {submitted_metrics['swaps']}, ops = {submitted_metrics['ops']}")
preview_score(submitted, submitted_energy, exact8)

### Rebuild it from scratch (optional, ~3 minutes)

Set `REOPTIMIZE = True` to rerun the 56-parameter COBYLA optimization.
The result should match the QASM above to the last printed digit.

In [ ]:
REOPTIMIZE = False

if REOPTIMIZE:
    rebuilt = solve_brickwall(H8, layers=6, maxiter=2500)
    print(f"rebuilt energy = {rebuilt['energy']:.10f} Ha")
    print(f"rebuilt error  = {rebuilt['error']:.6e} Ha")
    print(f"rebuilt score  = {rebuilt['score']:,}")
else:
    print("REOPTIMIZE=False — using the embedded QASM (4,392).")
    print("Set REOPTIMIZE=True to rerun the 56-parameter optimization.")

## 4. The hardware-optimal circuit — one Givens rotation, 4,386 noiseless

The exact state is HF + a single q3→q4 excitation. A single
`CX(3,4) · RY(θ,3) · CX(3,4)` prepares the optimal 2D subspace state:

- 4 X (HF preparation) + 2 CX + 1 RY, **depth 4**
- error **7.75e-4 Ha**, $A^2 = 0.9969$
- score **4,386** — only 6 points below the 6-layer circuit, with 10× fewer
  two-qubit gates.

On hardware this is the difference between **251** and **4,145**.

In [ ]:
givens = solve_givens(H8)
print(f"theta  = {givens['params'][0]:.8f}")
print(f"energy = {givens['energy']:.10f} Ha")
print(f"error  = {givens['error']:.6e} Ha")
print(f"depth  = {givens['metrics']['depth']}, swaps = {givens['metrics']['swaps']}, "
      f"ops = {givens['metrics']['ops']}")
preview_score(givens["circuit"], givens["energy"], exact8)

## 5. Side-by-side comparison

All circuits are evaluated on the same 8-qubit Hamiltonian. The HF row shows
that the starter's optimizer is not even beating the trivial reference state.

In [ ]:
def table_row(name, circuit):
    e = statevector_energy(circuit, H8)
    m = circuit_metrics(circuit)
    p = score_parts(8, abs(e - exact8), m["depth"], m["swaps"])
    return name, m["num_qubits"], m["depth"], m["swaps"], abs(e - exact8), p["score"]


hf_circuit = QuantumCircuit(8)
hf_circuit.x(range(4))
TWO_LAYER_QASM = """OPENQASM 2.0;
include "qelib1.inc";
qreg q[8];
ry(3.1415888798107083) q[0];
ry(3.8849862770085304) q[1];
ry(3.142316611485431) q[2];
ry(3.552136410142722) q[3];
ry(0.0005113165056188703) q[4];
ry(0.2972087377810939) q[5];
ry(-2.3030818000444962e-05) q[6];
ry(1.1393288402543815) q[7];
cx q[0],q[1];
cx q[2],q[3];
cx q[4],q[5];
cx q[6],q[7];
ry(-0.07957813494778965) q[0];
ry(-1.6815858146764546) q[1];
ry(1.5712666742262604) q[2];
ry(0.6826798429258022) q[3];
ry(0.4165963685935953) q[4];
ry(-0.29721253720513713) q[5];
ry(-0.362799828841251) q[6];
ry(-0.4457455898466569) q[7];
cx q[1],q[2];
cx q[3],q[4];
cx q[5],q[6];
ry(0.07959368856018191) q[0];
ry(0.7162372110756106) q[1];
ry(-1.569058245201401) q[2];
ry(3.028678523067778) q[3];
ry(-0.43085343829248757) q[4];
ry(-1.7597994644691542e-05) q[5];
ry(0.36277871845304227) q[6];
ry(-0.6935578946941612) q[7];"""
two_layer = from_qasm(TWO_LAYER_QASM)

rows = [
    table_row("HF only", hf_circuit),
    table_row("brickwall 2 layers", two_layer),
    table_row("brickwall 6 layers", submitted),
    table_row("Givens 2 CX", givens["circuit"]),
]

print(f"{'circuit':<22}{'N':>3}{'depth':>7}{'SWAP':>6}{'error':>12}{'score':>8}")
print("-" * 58)
for name, n, d, sw, err, sc in rows:
    print(f"{name:<22}{n:>3}{d:>7}{sw:>6}{err:>12.3e}{sc:>8,}")

## 6. With noise — the same circuits on `ibm_marrakesh`

Recorded with `EstimatorV2`, 4,096 shots, readout mitigation
(`resilience_level=1`), on the 8-qubit line
`[149, 148, 147, 137, 127, 126, 125, 124]` (zero SWAPs after transpiling).

| Circuit | 2q gates | Hardware energy | Error | Hardware score |
|---|---|---|---|---|
| 6-layer brickwall | 21 CZ | −111.7626694 | 0.3806 Ha | **251** |
| 2-layer brickwall | 7 CZ | −111.9994128 | 0.1438 Ha | **2,233** |
| **2-CX Givens** | **2 CZ** | **−112.1285148** | **0.0147 Ha** | **4,145** |

The Hamiltonian's large $Z$ coefficients (~0.85) amplify every bit flip, so
the noiseless-optimal circuit is the worst one on hardware. The 2-CX circuit
keeps 94.2% of its noiseless score.

ZNE (`resilience_level=2`, noise factors 1/3/5) returned `NaN` from the
exponential fit; its raw points were −112.12298, −112.11255, −112.08032,
and a linear extrapolation gives ~4,233. More shots would tighten that.

**Job IDs:** `dbji6b1ffo6c73b9nbmg` (6-layer),
`dbjj61pkdqps7390mvd0` (2-layer), `dbjj71hffo6c73b9p360` (Givens),
`dbjj8k1ffo6c73b9p5a0` (ZNE).

In [ ]:
RUN_HARDWARE = False  # set True to spend IBM Quantum time

BACKEND = "ibm_marrakesh"
QUBIT_LINE = [149, 148, 147, 137, 127, 126, 125, 124]

if RUN_HARDWARE:
    from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as Estimator
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

    keys = load_keys()
    service = QiskitRuntimeService(channel="ibm_quantum_platform",
                                   token=keys["IBM_Key"], instance=keys["IBM_Crn"])
    backend = service.backend(BACKEND)
    pm = generate_preset_pass_manager(optimization_level=3, backend=backend,
                                      initial_layout=QUBIT_LINE, seed_transpiler=42)
    isa = pm.run(givens["circuit"])
    H_isa = H8.apply_layout(isa.layout)

    estimator = Estimator(mode=backend)
    estimator.options.default_shots = 4096
    estimator.options.resilience_level = 1
    job = estimator.run([(isa, H_isa)])
    print("job id:", job.job_id())

    e_noisy = float(np.asarray(job.result()[0].data.evs).reshape(-1)[0])
    err = abs(e_noisy - exact8)
    A = max(0.0, 1.0 - err / 0.5)
    print(f"hardware energy = {e_noisy:.10f} Ha")
    print(f"error           = {err:.6e} Ha")
    print(f"hardware score  = {round(4400 * A * A):,}")
else:
    print("RUN_HARDWARE=False — recorded result: 4,145 points.")
    print("Set RUN_HARDWARE=True to rerun the 2-CX circuit on ibm_marrakesh.")

## 7. Submission

The official submission was made with the embedded 6-layer QASM and
`obtainedResult = 0.99911744`:

- **Submission ID:** `5571d09f-3e27-45b7-be28-35df1c7d1137`
- **Official score:** **4,392 points**
- The platform scores the QASM shape (N, depth, SWAPs) plus the reported $A$;
  it does not execute the circuit. That is why the official number is the
  noiseless one, while the same circuit scores 251 on real hardware.

In [ ]:
SUBMIT = False  # set True to send a new submission (10-second rate limit)

if SUBMIT:
    import requests

    keys = load_keys()
    payload = {
        "qasm": to_qasm(submitted),
        "challengeId": "challenge_intermediate",
        "obtainedResult": float(submitted_parts["accuracy"]),
    }
    response = requests.post(keys["Submission_URL"], json=payload,
                             headers={"X-API-Key": keys["Team_Key"]}, timeout=60)
    print(response.status_code, response.json())
else:
    print("SUBMIT=False — official result on record: 4,392 points.")

## 8. Self-contained notebook

The algorithm module and both QASM circuits are embedded in the cells above,
so this notebook runs on its own. The only external file is the optional,
gitignored `.env` used by the hardware/submission cells.

**Bottom line.** The provided Hamiltonian's ground state is 98.3% HF. The
starter's `real_amplitudes` ansatz cannot express the small correlation, and
COBYLA(200) returns an energy worse than HF. A 6-layer even/odd brickwall
reaches **4,392/4,400** noiselessly; a 2-CX Givens rotation reaches **4,386**
and is the only circuit that survives real hardware (**4,145**). The official
platform score is **4,392**.